# JazzCash Fraud Detection - July 2025 Analysis

This notebook documents the end-to-end process for analyzing JazzCash transaction data and fraud events for July 2025 using PySpark and PostgreSQL.

## Key Steps and Workflow

- **Spark Session Setup:**  
    Configures an optimized Spark session for large-scale data processing, including JDBC connectivity to the PostgreSQL database.

- **Data Loading:**  
    Loads the July 2025 transaction data from the `public.stixor_iar_jul` table using date-based partitioning for efficient reads.

- **Customer Segmentation:**  
    Identifies unique sending (`ac_from`) and receiving (`ac_to`) customers, and saves these lists to Parquet for further analysis.

- **Account Type Classification:**  
    Segregates accounts into "Customer Account" and "Non-Customer Account" using the `public.stixor_mbar_v` table, and joins these classifications with July sender/receiver lists.

- **Fraud Table Analysis:**  
    Loads and explores the `public.fraud` table, including metrics such as average resolution time and unique counts for key fraud-related columns.

- **Account Type Distribution for Fraud Entities:**  
    For each fraud-related entity (`complaint_msisdn`, `fraud_msisdn`, `victim_msisdn`, `ac_from`, `ac_to`), retrieves and summarizes account type distributions.

- **Feature Engineering:**  
    Provides SQL and PySpark code templates for generating time-windowed transaction features for downstream modeling.

## Outputs

- Parquet files containing distinct customer and non-customer account references for senders and receivers.
- Parquet files with account type distributions for fraud-related entities.
- Aggregated statistics and feature templates for advanced fraud analytics.

---
This notebook is designed for scalable, reproducible fraud analytics and feature engineering on JazzCash transaction data.

# Spark Sessions

In [1]:
from pyspark.sql import SparkSession
from datetime import datetime, timedelta
import os

# Database configuration
DB_CONFIG = {
    'host': '10.205.161.118',
    'port': '5432',
    'database': 'db_fraud',
    'user': 'dfstechbi',
    'password': 'DfsTeChB1@923'
}

# JDBC Configuration
jdbc_driver_path = "/root/research-dir/dev/jazzcash-fraud-detection/utils/postgresql-42.7.1.jar"
jdbc_url = f"jdbc:postgresql://{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"

# Optimized JDBC properties
properties = {
    "user": DB_CONFIG['user'],
    "password": DB_CONFIG['password'],
    "driver": "org.postgresql.Driver",
    "fetchsize": "10000",
    "batchsize": "15000",
    "isolationLevel": "READ_UNCOMMITTED",
    "queryTimeout": "1200",
    "loginTimeout": "60",
    "socketTimeout": "1200",
    "tcpKeepAlive": "true",
    "prepareThreshold": "5",
    "reWriteBatchedInserts": "true",
    "defaultRowFetchSize": "10000"
}

print("🚀 Creating optimized Spark session...")

# Create Spark session with comprehensive configuration
spark = SparkSession.builder \
    .appName("Load-IAR-July-Data-Optimized") \
    .master("spark://localhost:7077") \
    .config("spark.jars", jdbc_driver_path) \
    .config("spark.executor.memory", "20g") \
    .config("spark.executor.memoryOverhead", "1g") \
    .config("spark.driver.memory", "2g") \
    .config("spark.driver.memoryOverhead", "2g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "80") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "10000") \
    .config("spark.network.timeout", "800s") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .config("spark.sql.broadcastTimeout", "600s") \
    .getOrCreate()

# Set log level to reduce noise
spark.sparkContext.setLogLevel("WARN")

print("✅ Spark session created successfully!")
print(f"📱 Application ID: {spark.sparkContext.applicationId}")
print(f"🎯 Master: {spark.sparkContext.master}")

🚀 Creating optimized Spark session...


25/10/16 14:48:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/16 14:48:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


✅ Spark session created successfully!
📱 Application ID: app-20251016144824-0063
🎯 Master: spark://localhost:7077


# IAR Table July

In [ ]:
table_name = "public.stixor_iar_jul"

print("🔧 Setting up date-based predicates...")

# Create predicates for optimized partitioning
predicates = []
start_date = datetime.strptime("2025-07-01", "%Y-%m-%d")
end_date = datetime.strptime("2025-07-31", "%Y-%m-%d")

current_date = start_date
while current_date <= end_date:
    date_str = current_date.strftime("%Y-%m-%d")
    predicates.append(f"data_date = '{date_str}'")
    current_date += timedelta(days=1)

print(f"📊 Created {len(predicates)} predicates for date range")

print("\n🔗 Loading data from PostgreSQL...")

# Load data using predicate-based partitioning
df = spark.read.jdbc(
    url=jdbc_url,
    table=table_name,
    predicates=predicates,
    properties=properties
)

print("✅ Data loading configured successfully!")
print(f"📈 DataFrame partitions: {df.rdd.getNumPartitions()}")

# Cache for better performance
df.cache()
print("💾 DataFrame cached for optimized access")

print("\n📋 Basic DataFrame Info:")
print(f"🔢 Total partitions: {df.rdd.getNumPartitions()}")
print("📊 Schema preview:")
for field in df.schema.fields[:5]:
    print(f"  - {field.name}: {field.dataType}")

print(f"\n🎯 Ready to process fraud data for July 2025!")

In [2]:
df.groupBy('data_date').count().show()

+----------+--------+
| data_date|   count|
+----------+--------+
|2025-07-01|12546035|
|2025-07-02|13204748|
|2025-07-03|13375098|
|2025-07-04|12570037|
|2025-07-05|11150252|
|2025-07-06| 8757251|
|2025-07-07|13467970|
|2025-07-08|13363032|
|2025-07-09|13372656|
|2025-07-10|13322670|
|2025-07-11|13379387|
|2025-07-12|13532458|
|2025-07-13|12765033|
|2025-07-14|13706248|
|2025-07-15|13677842|
|2025-07-16|13804209|
|2025-07-17|13576565|
|2025-07-18|12923723|
|2025-07-19|13366922|
|2025-07-20|12659197|
+----------+--------+
only showing top 20 rows


## unique sending customers in July

In [ ]:
import time 
start_time = time.time()
df_july_senders = df.select("ac_from").distinct()
df_july_senders.cache()
july_senders_count = df_july_senders.count()
print(f"{july_senders_count:,} unique sending customers in July")
print(f"Completed in {(time.time() - start_time)/60:.2f} minutes\n")

22,947,201 unique sending customers in July
Completed in 1429.51 seconds



In [5]:
print(f"Number of partitions in df_july_senders: {df_july_senders.rdd.getNumPartitions()}")

Number of partitions in df_july_senders: 200


In [6]:
df_july_senders = df_july_senders.coalesce(23)

# OPTION A: Write to Parquet (recommended for further processing)
output_path = "../data/sender_customer_july_2025"

df_july_senders.write \
    .mode("overwrite") \
    .parquet(output_path)

## unique receiving customers in July

In [8]:
import time 
start_time = time.time()
df_july_receivers = df.select("ac_to").distinct()
df_july_receivers.cache()
july_receiver_count = df_july_receivers.count()
print(f"{july_receiver_count:,} unique receiving customers in July")
print(f"Completed in {(time.time() - start_time)/60:.2f} minutes\n")

19,452,571 unique receiving customers in July
Completed in 0.78 minutes



In [9]:
print(f"Number of partitions in df_july_receivers: {df_july_receivers.rdd.getNumPartitions()}")

Number of partitions in df_july_receivers: 200


In [10]:
df_july_receivers = df_july_receivers.coalesce(20)

output_path = "../data/receiver_customer_july_2025"

df_july_receivers.write \
    .mode("overwrite") \
    .parquet(output_path)

# Account Types for July Transaction Data

## July Senders (ac_from)

In [3]:
import math
from pyspark.sql.functions import col

stixor_mbar_v_table = "public.stixor_mbar_v"
df_july_senders = spark.read.parquet("../data/sender_customer_july_2025")
july_senders_count = df_july_senders.count()
print(f"📊 Total July senders loaded from parquet: {july_senders_count:,}")

📊 Total July senders loaded from parquet: 22,947,201


## MBar Customer Accounts

In [10]:
# Use SQL pushdown to filter account_type_name in the JDBC query itself
sql_query_customer = f"""
(SELECT DISTINCT a_c_reference 
    FROM {stixor_mbar_v_table} 
    WHERE account_type_name = 'Customer Account'
) AS customer_accounts
"""

df_mbar_customer_a_c_reference = spark.read.jdbc(
    url=jdbc_url,
    table=sql_query_customer,
    properties=properties
).distinct()

output_path_mbar_customer_a_c_reference = "../data/mbar_customer_a_c_reference"
df_mbar_customer_a_c_reference.write.mode("overwrite").parquet(output_path_mbar_customer_a_c_reference)

print(f"✅ Saved distinct a_c_reference for Customer Account to {output_path_mbar_customer_a_c_reference}")

✅ Saved distinct a_c_reference for Customer Account to ../data/mbar_customer_a_c_reference


In [5]:
output_path_mbar_customer_a_c_reference = "../data/mbar_customer_a_c_reference"
df_mbar_customer_a_c_reference = spark.read.parquet(output_path_mbar_customer_a_c_reference)
print(f"Count of df_mbar_customer_a_c_reference: {df_mbar_customer_a_c_reference.count():,}")

Count of df_mbar_customer_a_c_reference: 66,761,983


In [6]:
output_path_mbar_customer_a_c_reference = "../data/mbar_customer_a_c_reference"
df_mbar_customer_a_c_reference = spark.read.parquet(output_path_mbar_customer_a_c_reference)

df_july_customer_senders = df_july_senders.join(
    df_mbar_customer_a_c_reference,
    df_july_senders.ac_from == df_mbar_customer_a_c_reference.a_c_reference,
    how="inner"
).select(df_july_senders.ac_from)

print(f"✅ Joined July senders with Customer Account references. Result count: {df_july_customer_senders.count():,}")

✅ Joined July senders with Customer Account references. Result count: 22,579,490


In [15]:
df_july_customer_senders = df_july_customer_senders.coalesce(10)
output_path = "../data/july_2025_customer_senders"
df_july_customer_senders.write \
    .mode("overwrite") \
    .parquet(output_path)

In [10]:
output_path_mbar_customer_a_c_reference = "../data/mbar_customer_a_c_reference"
df_mbar_customer_a_c_reference = spark.read.parquet(output_path_mbar_customer_a_c_reference)

df_july_receivers = spark.read.parquet("../data/receiver_customer_july_2025")

df_july_customer_receivers = df_july_receivers.join(
    df_mbar_customer_a_c_reference,
    df_july_receivers.ac_to == df_mbar_customer_a_c_reference.a_c_reference,
    how="inner"
).select(df_july_receivers.ac_to)

print(f"✅ Joined July receivers with Customer Account references. Result count: {df_july_customer_receivers.count():,}")

✅ Joined July receivers with Customer Account references. Result count: 18,910,178


In [16]:
df_july_customer_receivers = df_july_customer_receivers.coalesce(10)
output_path = "../data/july_2025_customer_receivers"
df_july_customer_receivers.write \
    .mode("overwrite") \
    .parquet(output_path)

# MBar Non-customer Accounts

In [12]:
# Use SQL pushdown to filter account_type_name not equal to 'Customer Account'

stixor_mbar_v_table = "public.stixor_mbar_v"

sql_query_non_customer = f"""
(SELECT DISTINCT a_c_reference 
    FROM {stixor_mbar_v_table} 
    WHERE account_type_name <> 'Customer Account'
) AS non_customer_accounts
"""

df_mbar_non_customer_a_c_reference = spark.read.jdbc(
    url=jdbc_url,
    table=sql_query_non_customer,
    properties=properties
).distinct()

output_path_mbar_non_customer_a_c_reference = "../data/mbar_non_customer_a_c_reference"
df_mbar_non_customer_a_c_reference.write.mode("overwrite").parquet(output_path_mbar_non_customer_a_c_reference)

print(f"✅ Saved distinct a_c_reference for non-Customer Account to {output_path_mbar_non_customer_a_c_reference}")

In [13]:
output_path_mbar_non_customer_a_c_reference = "../data/mbar_non_customer_a_c_reference"
df_mbar_non_customer_a_c_reference = spark.read.parquet(output_path_mbar_non_customer_a_c_reference)

df_july_non_customer_senders = df_july_senders.join(
    df_mbar_non_customer_a_c_reference,
    df_july_senders.ac_from == df_mbar_non_customer_a_c_reference.a_c_reference,
    how="inner"
).select(df_july_senders.ac_from)

print(f"✅ Joined July senders with non-Customer Account references. Result count: {df_july_non_customer_senders.count():,}")b

✅ Joined July senders with non-Customer Account references. Result count: 285,089


In [17]:
df_july_non_customer_senders = df_july_non_customer_senders.coalesce(10)
output_path = "../data/july_2025_non_customer_senders"
df_july_non_customer_senders.write \
    .mode("overwrite") \
    .parquet(output_path)

In [14]:
output_path_mbar_non_customer_a_c_reference = "../data/mbar_non_customer_a_c_reference"
df_mbar_non_customer_a_c_reference = spark.read.parquet(output_path_mbar_non_customer_a_c_reference)

df_july_non_customer_receivers = df_july_receivers.join(
    df_mbar_non_customer_a_c_reference,
    df_july_receivers.ac_to == df_mbar_non_customer_a_c_reference.a_c_reference,
    how="inner"
).select(df_july_receivers.ac_to)

print(f"✅ Joined July receivers with non-Customer Account references. Result count: {df_july_non_customer_receivers.count():,}")

✅ Joined July receivers with non-Customer Account references. Result count: 469,387


In [18]:
df_july_non_customer_receivers = df_july_non_customer_receivers.coalesce(10)
output_path = "../data/july_2025_non_customer_receivers"
df_july_non_customer_receivers.write \
    .mode("overwrite") \
    .parquet(output_path)

# Fraud Table

In [11]:
fraud_table_name = "public.fraud"

df_fraud = spark.read.jdbc(
    url=jdbc_url,
    table=fraud_table_name,
    properties=properties
)

print("✅ Loaded public.fraud table")
df_fraud.show(5)

✅ Loaded public.fraud table


+-------------+-----------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+-----------+-------------+-------------------+-------------------+--------------------+
|complaint_num|   trans_id|    complaint_msisdn|        fraud_msisdn|       victim_msisdn|             ac_from|               ac_to|trx_amount|trx_channel|     trx_type|   created_datetime|  resolved_datetime|transaction_datetime|
+-------------+-----------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+-----------+-------------+-------------------+-------------------+--------------------+
|   COM2176544|74871448252|fvdT0uX2/YoR6/D2K...|wmQJL8FEmQvt6rPU8...|BASO2pY/4oQAtzaqQ...|BASO2pY/4oQAtzaqQ...|fvdT0uX2/YoR6/D2K...|      3900|        API|Transfer(C2C)|2025-02-11 13:06:06|2025-02-11 13:30:00| 2025-02-08 22:53:55|
|   COM2284237|76205288132|8xhAAusuCbt8bgI7z...|wmQJL8FEmQvt6rPU8...|8xhAAus

In [ ]:
from pyspark.sql.functions import avg, unix_timestamp

# Calculate average resolution time using transaction_datetime
df_fraud_with_resolution = df_fraud.filter(df_fraud.resolved_datetime.isNotNull())
avg_resolution_seconds = df_fraud_with_resolution.select(
    avg(unix_timestamp("resolved_datetime") - unix_timestamp("transaction_datetime")).alias("avg_resolution_seconds")
).collect()[0]["avg_resolution_seconds"]

avg_resolution_hours = avg_resolution_seconds / 3600 if avg_resolution_seconds is not None else None
print(f"Average resolution time: {avg_resolution_hours:.2f} hours" if avg_resolution_hours is not None else "No resolved cases found.")

Average resolution time: 103.13 hours


In [13]:
# Get distinct values for each column in df_fraud
distinct_complaint = df_fraud.select("complaint_msisdn").distinct()
distinct_fraud = df_fraud.select("fraud_msisdn").distinct()
distinct_victim = df_fraud.select("victim_msisdn").distinct()
distinct_ac_from = df_fraud.select("ac_from").distinct()
distinct_ac_to = df_fraud.select("ac_to").distinct()

# Save each to parquet
distinct_complaint.write.mode("overwrite").parquet("../data/distinct_complaint_msisdn")
distinct_fraud.write.mode("overwrite").parquet("../data/distinct_fraud_msisdn")
distinct_victim.write.mode("overwrite").parquet("../data/distinct_victim_msisdn")
distinct_ac_from.write.mode("overwrite").parquet("../data/distinct_ac_from")
distinct_ac_to.write.mode("overwrite").parquet("../data/distinct_ac_to")

# Print unique and total counts
print(f"complaint_msisdn: unique={distinct_complaint.count():,}, total={df_fraud.select('complaint_msisdn').count():,}")
print(f"fraud_msisdn: unique={distinct_fraud.count():,}, total={df_fraud.select('fraud_msisdn').count():,}")
print(f"victim_msisdn: unique={distinct_victim.count():,}, total={df_fraud.select('victim_msisdn').count():,}")
print(f"ac_from: unique={distinct_ac_from.count():,}, total={df_fraud.select('ac_from').count():,}")
print(f"ac_to: unique={distinct_ac_to.count():,}, total={df_fraud.select('ac_to').count():,}")

complaint_msisdn: unique=20,604, total=40,062
fraud_msisdn: unique=7,204, total=40,062
victim_msisdn: unique=20,598, total=40,062
ac_from: unique=20,581, total=40,062
ac_to: unique=14,862, total=40,062


# Account Types

## ac_from

In [27]:
stixor_mbar_v_table = "public.stixor_mbar_v"
ac_from_list = [row.ac_from for row in distinct_ac_from.collect()]
print(f"📊 Found {len(ac_from_list)} distinct ac_from values to filter")

ac_from_list_clean = [str(ac).replace("'", "''") for ac in ac_from_list]  # Escape single quotes
ac_from_in_clause = "'" + "','".join(ac_from_list) + "'"

sql_query = f"""
(SELECT a_c_reference, account_type_name 
    FROM {stixor_mbar_v_table} 
    WHERE a_c_reference IN ({ac_from_in_clause})) as filtered_mbar
"""

# Create optimized properties for the filtered query
filtered_properties = {
    **properties,  # Inherit base properties
    "fetchsize": "5000",           # Smaller fetch size for filtered data
    "batchsize": "10000",          # Optimized batch size
    "queryTimeout": "600",         # Shorter timeout for filtered query
    "socketTimeout": "600",        # Match query timeout
    "defaultRowFetchSize": "5000", # Optimized fetch size
    "prepareThreshold": "1",       # Prepare statement immediately
    "reWriteBatchedInserts": "true"
}

print(f"🚀 Loading filtered data with SQL pushdown...")

df_mbar_filtered = spark.read.jdbc(
    url=jdbc_url,
    table=sql_query,
    properties=filtered_properties
)

# Save results for future use
print("\n💾 Saving ac_from results...")
df_mbar_filtered.coalesce(5).write.mode("overwrite").parquet("../data/ac_from_accounts_with_types")
print("✅ Results saved to ../data/ac_from_accounts_with_types")


# Group by account_type_name and count the number of records for each type
df_mbar_filtered.groupBy("account_type_name").count().orderBy("count", ascending=False).show()


📊 Found 20581 distinct ac_from values to filter
🚀 Loading filtered data with SQL pushdown...

💾 Saving ac_from results...


25/10/15 17:34:55 WARN DAGScheduler: Broadcasting large task binary with size 1326.7 KiB


✅ Results saved to ../data/ac_from_accounts_with_types


25/10/15 17:34:55 WARN DAGScheduler: Broadcasting large task binary with size 1660.0 KiB


+--------------------+-----+
|   account_type_name|count|
+--------------------+-----+
|    Customer Account|20223|
|Organization Account|  147|
|Utility Bills Acc...|   34|
|Payment Gateway A...|   31|
|                NULL|   10|
| Account for Alfalah|    1|
+--------------------+-----+



25/10/15 17:34:56 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB


## ac_to

In [26]:
stixor_mbar_v_table = "public.stixor_mbar_v"
ac_to_list = [row.ac_to for row in distinct_ac_to.collect()]
print(f"📊 Found {len(ac_to_list)} distinct ac_to values to filter")

ac_to_list_clean = [str(ac).replace("'", "''") for ac in ac_to_list]  # Escape single quotes
ac_to_in_clause = "'" + "','".join(ac_to_list_clean) + "'"

sql_query_to = f"""
(SELECT a_c_reference, account_type_name 
    FROM {stixor_mbar_v_table} 
    WHERE a_c_reference IN ({ac_to_in_clause})) as filtered_mbar_to
"""

filtered_properties_to = {
    **properties,
    "fetchsize": "5000",
    "batchsize": "10000",
    "queryTimeout": "600",
    "socketTimeout": "600",
    "defaultRowFetchSize": "5000",
    "prepareThreshold": "1",
    "reWriteBatchedInserts": "true"
}

print(f"🚀 Loading filtered ac_to data with SQL pushdown...")

df_mbar_filtered_to = spark.read.jdbc(
    url=jdbc_url,
    table=sql_query_to,
    properties=filtered_properties_to
)

# Save results for future use
print("\n💾 Saving ac_to results...")
df_mbar_filtered_to.coalesce(5).write.mode("overwrite").parquet("../data/ac_to_accounts_with_types")
print("✅ Results saved to ../data/ac_to_accounts_with_types")

df_mbar_filtered_to.groupBy("account_type_name").count().orderBy("count", ascending=False).show()


📊 Found 14862 distinct ac_to values to filter
🚀 Loading filtered ac_to data with SQL pushdown...

💾 Saving ac_to results...


25/10/15 17:34:10 WARN DAGScheduler: Broadcasting large task binary with size 1025.2 KiB


✅ Results saved to ../data/ac_to_accounts_with_types


25/10/15 17:34:10 WARN DAGScheduler: Broadcasting large task binary with size 1207.6 KiB


+--------------------+-----+
|   account_type_name|count|
+--------------------+-----+
|    Customer Account|13887|
|Payment Gateway A...|  325|
|Utility Bills Acc...|  310|
|Organization Account|  268|
|                NULL|    5|
|SP Account for Is...|    1|
|Raast Settlement ...|    1|
|SP Account for Ac...|    1|
|1-Link Organizati...|    1|
+--------------------+-----+



25/10/15 17:34:11 WARN DAGScheduler: Broadcasting large task binary with size 1605.3 KiB


## fraud_msisdn

In [25]:
fraud_msisdn_list = [row.fraud_msisdn for row in distinct_fraud.collect()]
print(f"📊 Found {len(fraud_msisdn_list)} distinct fraud_msisdn values to filter")

fraud_msisdn_list_clean = [str(ac).replace("'", "''") for ac in fraud_msisdn_list]
fraud_msisdn_in_clause = "'" + "','".join(fraud_msisdn_list_clean) + "'"

sql_query_fraud = f"""
(SELECT a_c_reference, account_type_name 
    FROM {stixor_mbar_v_table} 
    WHERE a_c_reference IN ({fraud_msisdn_in_clause})) as filtered_mbar_fraud
"""

filtered_properties_fraud = {
    **properties,
    "fetchsize": "5000",
    "batchsize": "10000",
    "queryTimeout": "600",
    "socketTimeout": "600",
    "defaultRowFetchSize": "5000",
    "prepareThreshold": "1",
    "reWriteBatchedInserts": "true"
}

print(f"🚀 Loading filtered fraud_msisdn data with SQL pushdown...")

df_mbar_filtered_fraud = spark.read.jdbc(
    url=jdbc_url,
    table=sql_query_fraud,
    properties=filtered_properties_fraud
)
# Save results for future use
print("\n💾 Saving fraud_msisdn results...")
df_mbar_filtered_fraud.coalesce(5).write.mode("overwrite").parquet("../data/fraud_accounts_with_types")
print("✅ Results saved to ../data/fraud_accounts_with_types")

df_mbar_filtered_fraud.groupBy("account_type_name").count().orderBy("count", ascending=False).show()



📊 Found 7204 distinct fraud_msisdn values to filter
🚀 Loading filtered fraud_msisdn data with SQL pushdown...

💾 Saving fraud_msisdn results...
✅ Results saved to ../data/fraud_accounts_with_types
✅ Results saved to ../data/fraud_accounts_with_types
+--------------------+-----+
|   account_type_name|count|
+--------------------+-----+
|    Customer Account| 5276|
|Organization Account|   70|
|Payment Gateway A...|   63|
|                NULL|   57|
|Utility Bills Acc...|   51|
|Raast Settlement ...|    1|
|1-Link Organizati...|    1|
+--------------------+-----+

+--------------------+-----+
|   account_type_name|count|
+--------------------+-----+
|    Customer Account| 5276|
|Organization Account|   70|
|Payment Gateway A...|   63|
|                NULL|   57|
|Utility Bills Acc...|   51|
|Raast Settlement ...|    1|
|1-Link Organizati...|    1|
+--------------------+-----+



## victim_msisdn

In [23]:
# Analysis for victim_msisdn - Account types of fraud victims
victim_msisdn_list = [row.victim_msisdn for row in distinct_victim.collect()]
print(f"📊 Found {len(victim_msisdn_list)} distinct victim_msisdn values to filter")

# Handle potential SQL injection and optimize for large lists
victim_msisdn_list_clean = [str(msisdn).replace("'", "''") for msisdn in victim_msisdn_list]
victim_msisdn_in_clause = "'" + "','".join(victim_msisdn_list_clean) + "'"

sql_query_victim = f"""
(SELECT a_c_reference, account_type_name 
    FROM {stixor_mbar_v_table} 
    WHERE a_c_reference IN ({victim_msisdn_in_clause})) as filtered_mbar_victim
"""

# Create optimized properties for the filtered query
filtered_properties_victim = {
    **properties,  # Inherit base properties
    "fetchsize": "5000",           # Smaller fetch size for filtered data
    "batchsize": "10000",          # Optimized batch size
    "queryTimeout": "600",         # Shorter timeout for filtered query
    "socketTimeout": "600",        # Match query timeout
    "defaultRowFetchSize": "5000", # Optimized fetch size
    "prepareThreshold": "1",       # Prepare statement immediately
    "reWriteBatchedInserts": "true"
}

print(f"🚀 Loading filtered victim_msisdn data with SQL pushdown...")
print(f"📋 Query preview: {sql_query_victim[:100]}...")

df_mbar_filtered_victim = spark.read.jdbc(
    url=jdbc_url,
    table=sql_query_victim,
    properties=filtered_properties_victim
)

print(f"✅ Loaded {df_mbar_filtered_victim.count():,} filtered records for victim_msisdn")

# Group by account_type_name and count the number of records for each type
print("\n📊 Account Type Distribution for victim_msisdn (Fraud Victims):")
df_mbar_filtered_victim.groupBy("account_type_name").count().orderBy("count", ascending=False).show()

# Save results for future use
print("\n💾 Saving victim_msisdn results...")
df_mbar_filtered_victim.coalesce(5).write.mode("overwrite").parquet("../data/victim_accounts_with_types")
print("✅ Results saved to ../data/victim_accounts_with_types")

📊 Found 20598 distinct victim_msisdn values to filter
🚀 Loading filtered victim_msisdn data with SQL pushdown...
📋 Query preview: 
(SELECT a_c_reference, account_type_name 
    FROM public.stixor_mbar_v 
    WHERE a_c_reference IN...


25/10/15 17:32:42 WARN DAGScheduler: Broadcasting large task binary with size 1103.0 KiB
25/10/15 17:32:43 WARN DAGScheduler: Broadcasting large task binary with size 1661.4 KiB
25/10/15 17:32:43 WARN DAGScheduler: Broadcasting large task binary with size 1661.4 KiB


✅ Loaded 20,365 filtered records for victim_msisdn

📊 Account Type Distribution for victim_msisdn (Fraud Victims):


25/10/15 17:32:44 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
25/10/15 17:32:44 WARN DAGScheduler: Broadcasting large task binary with size 1327.6 KiB


+--------------------+-----+
|   account_type_name|count|
+--------------------+-----+
|    Customer Account|20236|
|Organization Account|   52|
|Utility Bills Acc...|   32|
|Payment Gateway A...|   28|
|                NULL|   17|
+--------------------+-----+


💾 Saving victim_msisdn results...
✅ Results saved to ../data/victim_accounts_with_types
✅ Results saved to ../data/victim_accounts_with_types


## complaint_msisdn

In [24]:
# Analysis for complaint_msisdn - Account types of those who filed fraud complaints
complaint_msisdn_list = [row.complaint_msisdn for row in distinct_complaint.collect()]
print(f"📊 Found {len(complaint_msisdn_list)} distinct complaint_msisdn values to filter")

# Handle potential SQL injection and optimize for large lists
complaint_msisdn_list_clean = [str(msisdn).replace("'", "''") for msisdn in complaint_msisdn_list]
complaint_msisdn_in_clause = "'" + "','".join(complaint_msisdn_list_clean) + "'"

sql_query_complaint = f"""
(SELECT a_c_reference, account_type_name 
    FROM {stixor_mbar_v_table} 
    WHERE a_c_reference IN ({complaint_msisdn_in_clause})) as filtered_mbar_complaint
"""

# Create optimized properties for the filtered query
filtered_properties_complaint = {
    **properties,  # Inherit base properties
    "fetchsize": "5000",           # Smaller fetch size for filtered data
    "batchsize": "10000",          # Optimized batch size
    "queryTimeout": "600",         # Shorter timeout for filtered query
    "socketTimeout": "600",        # Match query timeout
    "defaultRowFetchSize": "5000", # Optimized fetch size
    "prepareThreshold": "1",       # Prepare statement immediately
    "reWriteBatchedInserts": "true"
}

print(f"🚀 Loading filtered complaint_msisdn data with SQL pushdown...")
print(f"📋 Query preview: {sql_query_complaint[:100]}...")

df_mbar_filtered_complaint = spark.read.jdbc(
    url=jdbc_url,
    table=sql_query_complaint,
    properties=filtered_properties_complaint
)

print(f"✅ Loaded {df_mbar_filtered_complaint.count():,} filtered records for complaint_msisdn")

# Group by account_type_name and count the number of records for each type
print("\n📊 Account Type Distribution for complaint_msisdn (Complaint Filers):")
df_mbar_filtered_complaint.groupBy("account_type_name").count().orderBy("count", ascending=False).show()

# Save results for future use
print("\n💾 Saving complaint_msisdn results...")
df_mbar_filtered_complaint.coalesce(5).write.mode("overwrite").parquet("../data/complaint_accounts_with_types")
print("✅ Results saved to ../data/complaint_accounts_with_types")

📊 Found 20604 distinct complaint_msisdn values to filter
🚀 Loading filtered complaint_msisdn data with SQL pushdown...
📋 Query preview: 
(SELECT a_c_reference, account_type_name 
    FROM public.stixor_mbar_v 
    WHERE a_c_reference IN...


25/10/15 17:32:53 WARN DAGScheduler: Broadcasting large task binary with size 1103.3 KiB
25/10/15 17:32:53 WARN DAGScheduler: Broadcasting large task binary with size 1661.9 KiB
25/10/15 17:32:53 WARN DAGScheduler: Broadcasting large task binary with size 1661.9 KiB


✅ Loaded 20,335 filtered records for complaint_msisdn

📊 Account Type Distribution for complaint_msisdn (Complaint Filers):


25/10/15 17:32:54 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
25/10/15 17:32:54 WARN DAGScheduler: Broadcasting large task binary with size 1328.0 KiB


+--------------------+-----+
|   account_type_name|count|
+--------------------+-----+
|    Customer Account|20195|
|Organization Account|   52|
|Utility Bills Acc...|   34|
|Payment Gateway A...|   30|
|                NULL|   24|
+--------------------+-----+


💾 Saving complaint_msisdn results...
✅ Results saved to ../data/complaint_accounts_with_types
✅ Results saved to ../data/complaint_accounts_with_types


# Features for Sample Accounts 

In [2]:
# Load July customer senders from saved parquet
df_july_customer_senders = spark.read.parquet("../data/july_2025_customer_senders")
print(f"📊 Loaded July customer senders: {df_july_customer_senders.count():,}")

📊 Loaded July customer senders: 22,579,490


In [7]:
# Load 1000 sample sender customer IDs and collect them as a list
sample_senders = df_july_customer_senders.limit(1)
sample_ac_from_list = [row.ac_from for row in sample_senders.collect()]
print(f"🎯 Selected sample of 10 customer senders")

🎯 Selected sample of 10 customer senders


In [8]:
# Create IN clause for SQL pushdown
ac_from_list_clean = [str(ac).replace("'", "''") for ac in sample_ac_from_list]
ac_from_in_clause = "'" + "','".join(ac_from_list_clean) + "'"

In [ ]:
# Create predicates for each sample account
predicates = [f"ac_from = '{ac}' AND data_date BETWEEN '2025-06-01' AND '2025-07-31'" for ac in sample_ac_from_list]

stixor_iar_table = "public.stixor_iar"

print(f"🔍 Loading filtered transactions using {len(predicates)} predicates...")

query_properties = {
    **properties,
    "fetchsize": "5000",
    "batchsize": "10000",
    "queryTimeout": "600",
    "socketTimeout": "600",
    "defaultRowFetchSize": "5000",
    "prepareThreshold": "1",
    "reWriteBatchedInserts": "true"
}
df_sample_tx = spark.read.jdbc(
    url=jdbc_url,
    table=stixor_iar_table,
    predicates=predicates,
    properties=query_properties
)

print(f"✅ Loaded filtered transactions: {df_sample_tx.count():,} rows")

🔍 Loading filtered transactions using 1 predicates...


# Featues

In [ ]:

# Define event date for feature windows
EVENT_DATE = "2025-07-31"
event_date = F.to_date(F.lit(EVENT_DATE))

# Add window flags and time features
windowed = (
    df_sample_tx
    .withColumn("data_date", F.to_date("data_date"))
    .withColumn("win_1d", F.col("data_date") >= F.date_sub(event_date, 1))
    .withColumn("win_3d", F.col("data_date") >= F.date_sub(event_date, 3))
    .withColumn("win_7d", F.col("data_date") >= F.date_sub(event_date, 7))
    .withColumn("win_15d", F.col("data_date") >= F.date_sub(event_date, 15))
    .withColumn("win_30d", F.col("data_date") >= F.date_sub(event_date, 30))
    .withColumn("trx_hour", F.hour("trans_initiate_time"))
    .withColumn(
        "trx_time_bucket",
        F.when((F.col("trx_hour") >= 0) & (F.col("trx_hour") < 6), "midnight")
        .when((F.col("trx_hour") >= 6) & (F.col("trx_hour") < 12), "morning")
        .when((F.col("trx_hour") >= 12) & (F.col("trx_hour") < 18), "afternoon")
        .otherwise("evening")
    )
)

# Generate window aggregations function
def generate_window_aggs(window_flag: str, hours: int):
    return [
        F.count(F.when(F.col(window_flag), True)).alias(f"tx_count_{window_flag}"),
        F.count(F.when(F.col(window_flag) & (F.col("trx_status") == "Completed"), True)).alias(f"tx_success_{window_flag}"),
        F.count(F.when(F.col(window_flag) & (F.col("trx_status") != "Completed"), True)).alias(f"tx_failed_{window_flag}"),
        F.countDistinct(F.when(F.col(window_flag), F.col("data_date"))).alias(f"active_days_{window_flag}"),
        F.sum(F.when(F.col(window_flag), F.col("trx_amt"))).alias(f"sum_trx_amt_{window_flag}"),
        F.avg(F.when(F.col(window_flag), F.col("trx_amt"))).alias(f"avg_trx_amt_{window_flag}"),
        F.max(F.when(F.col(window_flag), F.col("trx_amt"))).alias(f"max_trx_amt_{window_flag}"),
        F.min(F.when(F.col(window_flag), F.col("trx_amt"))).alias(f"min_trx_amt_{window_flag}"),
        F.stddev(F.when(F.col(window_flag), F.col("trx_amt"))).alias(f"stddev_trx_amt_{window_flag}"),
        F.countDistinct(F.when(F.col(window_flag), F.col("trx_channel"))).alias(f"unique_channels_{window_flag}"),
        F.countDistinct(F.when(F.col(window_flag), F.col("trx_type"))).alias(f"unique_types_{window_flag}"),
        F.countDistinct(F.when(F.col(window_flag), F.col("merchant_id"))).alias(f"unique_merchants_{window_flag}"),
        (F.count(F.when(F.col(window_flag) & (F.col("trx_status") != "Completed"), True)).cast("float") /
         F.when(F.count(F.when(F.col(window_flag), True)) != 0,
                F.count(F.when(F.col(window_flag), True)))).alias(f"failure_ratio_{window_flag}"),
        F.count(F.when(F.col(window_flag) & (F.col("trx_amt") > 100000), True)).alias(f"high_value_count_{window_flag}"),
        (F.count(F.when(F.col(window_flag) & (F.col("trx_amt") > 100000), True)).cast("float") /
         F.when(F.count(F.when(F.col(window_flag), True)) != 0,
                F.count(F.when(F.col(window_flag), True)))).alias(f"high_value_ratio_{window_flag}")
    ]

# Create aggregations for all windows
aggregations = []
windows = [("win_1d", 24), ("win_3d", 72), ("win_7d", 168), ("win_15d", 360), ("win_30d", 720)]
for win_flag, hrs in windows:
    aggregations.extend(generate_window_aggs(win_flag, hrs))

print(f"🔧 Creating {len(aggregations)} features across {len(windows)} time windows...")

# Compute features and save
features_agg = windowed.groupBy("ac_from").agg(*aggregations)

# Add customer type flag
features_agg = features_agg.withColumn("customer_type", F.lit("customer"))

print(f"✅ Generated features for {features_agg.count():,} customer accounts")
print(f"📊 Total feature columns: {len(features_agg.columns)}")

# Save features
output_path = "../data/features_july_customer_senders_1000_optimized/"
features_agg.coalesce(10).write.mode("overwrite").parquet(output_path)
print(f"💾 Features saved to {output_path}")

# Show sample features
print("\n📈 Sample features:")
features_agg.select("ac_from", "tx_count_win_1d", "tx_count_win_7d", "tx_count_win_30d", 
                   "avg_trx_amt_win_7d", "failure_ratio_win_7d").show(5)